# Phase 2 — Data Representation (Prototyping Surface)

This notebook is where we **prototype the SQL** for the readmission dataset and
analyze each layer before freezing it into Dataform. The rule: we iterate on SQL
*against BigQuery* (not in pandas), pull the **result** back for EDA, and the
final query text drops verbatim into the matching `.sqlx` — table refs wrapped
in `ref()`.

We build in **DAG order**, mirroring the Dataform build so what we see here
matches what ships:

```
sources -> cohort -> cohort_split -> features -> features_clean -> analytics_dataset
```

The split happens **before** feature engineering and missingness handling — any
train-derived statistic (e.g. an imputation median) must come from training rows
only. That ordering is the core leakage guard, so we honor it here too: never
profile distributions or compute fill values on the full cohort.

| Surface | Owns |
|---|---|
| This notebook | EDA, SQL prototyping, distribution/null profiling, deriving assertion thresholds |
| Dataform `.sqlx` | The frozen logic, `ref()` wiring, the DAG, the data contract |
| `docs/workflow.md` | Back-filling placeholders (transformation strategy, feature list) |

## 0. Setup — BigQuery connection & config

Source dataset names are the **verified** PhysioNet values
(`mimiciv_3_1_*`), matching `dataform.json`. The **billing project** is *not*
hardcoded — it resolves from `PROJECT_ID` (env var or the repo-root `.env`, the
same file the shell scripts read), so no personal project ID is committed.
See the Dataform README's *"Running this yourself"* section for first-time setup.

Queries read from the public `physionet-data` project but are **billed** to
`BILLING_PROJECT`.


In [26]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
from google.cloud import bigquery


def _resolve_billing_project() -> str:
    """Resolve the GCP billing project — no personal ID committed to git.

    Order: the PROJECT_ID env var, then the repo-root .env (the same file the
    shell scripts read — see .env.example). Kept dependency-free so the
    notebook runs without python-dotenv. See the Dataform README's
    "Running this yourself" section.
    """
    if os.environ.get("PROJECT_ID"):
        return os.environ["PROJECT_ID"]

    for directory in [Path.cwd(), *Path.cwd().resolve().parents]:
        env_file = directory / ".env"
        if env_file.exists():
            for line in env_file.read_text().splitlines():
                if line.strip().startswith("PROJECT_ID="):
                    return line.split("=", 1)[1].strip()
            break

    raise RuntimeError(
        "PROJECT_ID is not set. Export it, or copy .env.example -> .env at the "
        "repo root and set PROJECT_ID to your GCP project."
    )


# --- Billing / connection ------------------------------------------------
# MIMIC-IV lives in the public `physionet-data` project; we read from there
# but queries are billed to this project.
BILLING_PROJECT = _resolve_billing_project()

# --- MIMIC-IV source datasets (PhysioNet public project) -----------------
MIMIC_PROJECT = "physionet-data"
MIMIC_HOSP = f"{MIMIC_PROJECT}.mimiciv_3_1_hosp"
MIMIC_ICU = f"{MIMIC_PROJECT}.mimiciv_3_1_icu"
MIMIC_ED = f"{MIMIC_PROJECT}.mimiciv_ed"
MIMIC_NOTE = f"{MIMIC_PROJECT}.mimiciv_note"

client = bigquery.Client(project=BILLING_PROJECT)


def run_sql(sql: str) -> pd.DataFrame:
    """Run BigQuery SQL and return the result as a DataFrame.

    Prototype layer SQL here; once it's right, the final text drops into the
    matching Dataform .sqlx with table refs wrapped in ref().
    """
    return client.query(sql).result().to_dataframe()


print(f"BigQuery client ready - billing project: {client.project}")


BigQuery client ready - billing project: trim-icon-498815-a0


**Tier B — needs a derivation:**

| Rule | Why |
|---|---|
| Administrative transfers / contiguous stays | Compare each admission to the patient's prior one; a re-admit within `CONTIGUOUS_HOURS` of the previous discharge is the same episode. **Decision: keep the first `hadm_id`, drop the continuation.** |
| Planned follow-up visits | **Decision: exclude `admission_type IN ('ELECTIVE', 'SURGICAL SAME DAY ADMISSION')`** — the only two scheduled-in-advance types. |

**Observation stays:** kept. The four `*OBSERVATION*` types are treated as real
inpatient admissions; the **LOS ≥ 1 day** gate is what removes the trivial ones,
not a blanket type exclusion.

The cell below profiles the categorical vocabularies (already run); the two cells
after build the spine in two steps — a **funnel** (count dropped at each gate, for
visibility) then the **skinny spine** itself. The final spine SQL backs
`staging/cohort.sqlx`.

In [2]:
# Categorical vocabulary — counts per distinct value for the three columns
# that drive Tier B (planned-visit + transfer logic). Read-only.
vocab_sql = f"""
SELECT 'admission_type'     AS column_name, admission_type     AS value, COUNT(*) AS n
FROM `{MIMIC_HOSP}.admissions` GROUP BY value
UNION ALL
SELECT 'admission_location' AS column_name, admission_location AS value, COUNT(*) AS n
FROM `{MIMIC_HOSP}.admissions` GROUP BY value
UNION ALL
SELECT 'discharge_location' AS column_name, discharge_location AS value, COUNT(*) AS n
FROM `{MIMIC_HOSP}.admissions` GROUP BY value
ORDER BY column_name, n DESC
"""

vocab = run_sql(vocab_sql)
for col in ["admission_type", "admission_location", "discharge_location"]:
    print(f"\n=== {col} ===")
    print(vocab[vocab.column_name == col][["value", "n"]].to_string(index=False))


=== admission_type ===
                      value      n
                   EW EMER. 177459
             EU OBSERVATION 119456
          OBSERVATION ADMIT  84437
                     URGENT  54929
SURGICAL SAME DAY ADMISSION  42898
         DIRECT OBSERVATION  24551
               DIRECT EMER.  21973
                   ELECTIVE  13130
     AMBULATORY OBSERVATION   7195

=== admission_location ===
                                 value      n
                        EMERGENCY ROOM 244179
                    PHYSICIAN REFERRAL 163228
                TRANSFER FROM HOSPITAL  56227
                 WALK-IN/SELF REFERRAL  42365
                       CLINIC REFERRAL  12965
                        PROCEDURE SITE   8518
TRANSFER FROM SKILLED NURSING FACILITY   6317
    INTERNAL TRANSFER TO OR FROM PSYCH   5837
                                  PACU   5734
             INFORMATION NOT AVAILABLE    402
           AMBULATORY SURGERY TRANSFER    255
                                  None      1


In [27]:
# --- Spine parameters (tunable, kept visible) ----------------------------
PLANNED_TYPES = ["ELECTIVE", "SURGICAL SAME DAY ADMISSION"]
CONTIGUOUS_HOURS = 24  # re-admit within 1 day of prior discharge = same episode

_planned_list = ", ".join(f"'{t}'" for t in PLANNED_TYPES)

# Shared base CTE — reused by both the funnel (below) and the spine (next cell)
# so the two can never drift. `prev_dischtime` uses LAG over the patient's full
# admission timeline to detect contiguous stays.
BASE_CTE = f"""
WITH base AS (
  SELECT
    a.subject_id, a.hadm_id, a.admittime, a.dischtime,
    p.anchor_age, a.hospital_expire_flag, a.deathtime, a.admission_type,
    TIMESTAMP_DIFF(a.dischtime, a.admittime, HOUR) AS los_hours,
    LAG(a.dischtime) OVER (PARTITION BY a.subject_id ORDER BY a.admittime) AS prev_dischtime
  FROM `{MIMIC_HOSP}.admissions` AS a
  JOIN `{MIMIC_HOSP}.patients`   AS p USING (subject_id)
),
flagged AS (
  SELECT *,
    (prev_dischtime IS NOT NULL
       AND TIMESTAMP_DIFF(admittime, prev_dischtime, HOUR) <= {CONTIGUOUS_HOURS}
    ) AS is_continuation
  FROM base
)
"""

# Funnel: cumulative survivors after each gate is applied in order.
funnel_sql = BASE_CTE + f"""
SELECT
  COUNT(*) AS s0_all_admissions,
  COUNTIF(anchor_age >= 18) AS s1_adult,
  COUNTIF(anchor_age >= 18 AND hospital_expire_flag = 0 AND deathtime IS NULL) AS s2_alive,
  COUNTIF(anchor_age >= 18 AND hospital_expire_flag = 0 AND deathtime IS NULL
          AND los_hours >= 24) AS s3_los_ge_1d,
  COUNTIF(anchor_age >= 18 AND hospital_expire_flag = 0 AND deathtime IS NULL
          AND los_hours >= 24 AND admission_type NOT IN ({_planned_list})) AS s4_unplanned,
  COUNTIF(anchor_age >= 18 AND hospital_expire_flag = 0 AND deathtime IS NULL
          AND los_hours >= 24 AND admission_type NOT IN ({_planned_list})
          AND NOT is_continuation) AS s5_spine
FROM flagged
"""

funnel = run_sql(funnel_sql).T.rename(columns={0: "rows"})
funnel["dropped"] = funnel["rows"].diff().fillna(0).astype(int) * -1
print(funnel.to_string())

                     rows  dropped
s0_all_admissions  546028        0
s1_adult           546028        0
s2_alive           534227    11801
s3_los_ge_1d       418550   115677
s4_unplanned       364738    53812
s5_spine           352699    12039


In [4]:
# The skinny spine: reuse BASE_CTE (same gates as the funnel, so no drift).
# Identity + timing only — demographics/label join on in later layers.
spine_sql = BASE_CTE + f"""
SELECT
  subject_id,
  hadm_id,
  admittime,
  dischtime
FROM flagged
WHERE anchor_age >= 18                                   -- adult
  AND hospital_expire_flag = 0 AND deathtime IS NULL     -- discharged alive
  AND los_hours >= 24                                    -- LOS >= 1 day
  AND admission_type NOT IN ({_planned_list})            -- exclude planned
  AND NOT is_continuation                                -- collapse contiguous stays
"""

spine = run_sql(spine_sql)

# Integrity checks — must match the funnel's s5_spine and be unique on hadm_id.
print(f"spine rows         : {len(spine):,}  (expected 352,699)")
print(f"distinct hadm_id   : {spine.hadm_id.nunique():,}  (must equal rows)")
print(f"distinct subject_id: {spine.subject_id.nunique():,}  (patients)")
print(f"null check         : {spine.isnull().sum().sum()} nulls")
spine.head()

spine rows         : 352,699  (expected 352,699)
distinct hadm_id   : 352,699  (must equal rows)
distinct subject_id: 164,847  (patients)
null check         : 0 nulls


,subject_id,hadm_id,admittime,dischtime
0,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00
1,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00
3,10000084,23052089,2160-11-21 01:56:00,2160-11-25 14:52:00
4,10000117,27988844,2183-09-18 18:10:00,2183-09-21 16:30:00


## 2. Deterministic split

Assign each admission to one of five disjoint groups via a **weighted** hash on
`subject_id`: `MOD(ABS(FARM_FINGERPRINT(subject_id)), 100)` → a 0–99 bucket,
mapped to groups by range.

**Weights — 70 / 14 / 14 / 1 / 1:**

| Group | Buckets (0–99) | Share |
|---|---|---|
| validation | 0–13 | 14% |
| test | 14–27 | 14% |
| prod_test | 28 | 1% |
| demo | 29 | 1% |
| **train** | 30–99 | **70%** |

- **All five groups are listed explicitly** in `SPLIT_RANGES` (train included),
  and a coverage guard asserts the ranges cover every bucket 0–99 — so the shares
  provably sum to 100% with nothing unassigned. (Requested 68/14/14/1/1 sums to
  98%; the leftover 2% goes to train → 70%.)
- **Keyed on `subject_id`, not `hadm_id`** — all of a patient's admissions land
  in one group. Leakage guard from `workflow.md` §2, enforced by
  `assertions/split_is_disjoint.sqlx`.
- **Deterministic + reproducible** — `FARM_FINGERPRINT` is a stable hash; same
  assignment every run, no stored seed.

This backs `staging/cohort_split.sqlx`.

In [5]:
# --- Split parameters (visible) ------------------------------------------
# Weighted split: hash subject_id into 0..SPLIT_RESOLUTION-1, map ranges to
# groups. Ranges are listed in bucket order and must cover every bucket.
SPLIT_RESOLUTION = 100  # bucket granularity (100 -> 1% steps)

# (group, upper_exclusive_bound) — each group claims [prev_bound, bound).
# train is listed explicitly and closes out the range at SPLIT_RESOLUTION.
SPLIT_RANGES = [
    ("validation", 14),   # buckets  0..13  -> 14%
    ("test",       28),   # buckets 14..27  -> 14%
    ("prod_test",  29),   # bucket  28      ->  1%
    ("demo",       30),   # bucket  29      ->  1%
    ("train",      100),  # buckets 30..99  -> 70%
]

# Coverage guard: bounds must be strictly increasing and the last must equal
# SPLIT_RESOLUTION, so every bucket (including train's) is accounted for.
_bounds = [b for _, b in SPLIT_RANGES]
assert _bounds == sorted(set(_bounds)), "bounds must be strictly increasing"
assert _bounds[-1] == SPLIT_RESOLUTION, "last bound must close the full range"

# Build the CASE ladder from the ranges (so SQL + table above can't drift).
_when = "\n".join(
    f"      WHEN h < {bound} THEN '{name}'" for name, bound in SPLIT_RANGES
)
_split_case = f"    CASE\n{_when}\n    END AS split_name"

# Reuse BASE_CTE + the validated spine gates, then hash-bucket on subject_id.
split_sql = BASE_CTE + f""",
spine AS (
  SELECT subject_id, hadm_id, admittime, dischtime
  FROM flagged
  WHERE anchor_age >= 18
    AND hospital_expire_flag = 0 AND deathtime IS NULL
    AND los_hours >= 24
    AND admission_type NOT IN ({_planned_list})
    AND NOT is_continuation
),
hashed AS (
  SELECT
    s.*,
    MOD(ABS(FARM_FINGERPRINT(CAST(s.subject_id AS STRING))), {SPLIT_RESOLUTION}) AS h
  FROM spine AS s
)
SELECT
  subject_id, hadm_id, admittime, dischtime,
  h AS split_bucket,
{_split_case}
FROM hashed
"""

split = run_sql(split_sql)

# Proportions by group — admissions and distinct patients.
summary = (
    split.groupby("split_name")
    .agg(admissions=("hadm_id", "size"), patients=("subject_id", "nunique"))
    .assign(pct=lambda d: (d.admissions / d.admissions.sum() * 100).round(1))
    .sort_values("admissions", ascending=False)
)
print(summary.to_string())

# Leakage guard (mirrors assertions/split_is_disjoint.sqlx): every subject_id
# must map to exactly one group.
spanning = split.groupby("subject_id").split_name.nunique()
print(f"\nrows: {len(split):,}   subjects in >1 group: {(spanning > 1).sum()}  (must be 0)")
print(f"unassigned (NULL) split_name: {split.split_name.isnull().sum()}  (must be 0)")

            admissions  patients   pct
split_name                            
train           247687    115468  70.2
test             49103     23126  13.9
validation       48924     23002  13.9
prod_test         3583      1646   1.0
demo              3402      1605   1.0

rows: 352,699   subjects in >1 group: 0  (must be 0)
unassigned (NULL) split_name: 0  (must be 0)


## 3. Features

Built **after** the split, so no feature logic crosses the train/val/test
boundary. The six groups from `workflow.md` §3 each become their **own
intermediate model**, all keyed on `hadm_id`, that `features.sqlx` joins onto
`cohort_split`:

| Group | Source tables | `.sqlx` |
|---|---|---|
| Demographics & Admin | `patients`, `admissions` | `feat_demographics` |
| Historical Utilization | `admissions`, `edstays` | `feat_utilization` |
| Structured Clinical Codes | `diagnoses_icd`, `procedures_icd` | `feat_codes` |
| Medications | `prescriptions` | `feat_medications` |
| Physiology & Labs | `labevents`, `chartevents` | `feat_labs` |
| Unstructured Text Notes | `discharge`, `radiology` | `feat_notes` |

We build **one group at a time**, each following the same three steps:

1. **Locate** — profile the source columns (types, null rates, vocab) so we
   know what we're aggregating.
2. **Aggregate** — write the per-`hadm_id` SQL, joined onto the spine.
3. **Sanity check** — row count matches the spine (one row per admission),
   no unexpected nulls, value ranges plausible.

Every group's grain is **one row per `hadm_id`** so the joins in `features.sqlx`
stay 1:1 and can't fan out.


### 3a. Demographics & Admin

Baseline patient profile — physical vulnerability + socioeconomic resources.
Mostly **1:1 attributes**, not true aggregations, so the "aggregation" step is
really a select + a couple of derivations.

| Feature | Source | Note |
|---|---|---|
| `age` | `patients.anchor_age`, `admissions.admittime` | MIMIC stores age at `anchor_year`; age at admission = `anchor_age + (year(admittime) - anchor_year)`. |
| `gender` | `patients.gender` | M/F. |
| `marital_status` | `admissions.marital_status` | categorical, expect nulls. |
| `language` | `admissions.language` | categorical. |
| `race` | `admissions.race` | the "ethnicity" feature; column is `race` in v3.1. |
| `admission_type` | `admissions.admission_type` | already profiled in §1 vocab. |
| `insurance` | `admissions.insurance` | categorical. |
| `discharge_location` | `admissions.discharge_location` | categorical, expect nulls. |

**Step 1 — Locate.** Profile each source column's type, null rate, and (for the
categoricals) distinct-value count, restricted to the cohort so the null rates
reflect *our* population, not all of MIMIC. This tells us which columns need a
missingness strategy later (handled in `features_clean`, not here).


In [6]:
# Step 1 — Locate. Profile the demographics columns ON THE COHORT (reuse
# BASE_CTE + spine gates, then join patients/admissions) so null rates reflect
# our population, not all of MIMIC. One summary row, reshaped below.
demo_profile_sql = BASE_CTE + f""",
spine AS (
  SELECT subject_id, hadm_id, admittime
  FROM flagged
  WHERE anchor_age >= 18
    AND hospital_expire_flag = 0 AND deathtime IS NULL
    AND los_hours >= 24
    AND admission_type NOT IN ({_planned_list})
    AND NOT is_continuation
),
demo AS (
  SELECT
    s.hadm_id,
    p.gender,
    a.marital_status,
    a.language,
    a.race,
    a.insurance,
    a.discharge_location
  FROM spine AS s
  JOIN `{MIMIC_HOSP}.patients`   AS p USING (subject_id)
  JOIN `{MIMIC_HOSP}.admissions` AS a USING (hadm_id)
)
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT hadm_id) AS distinct_hadm,
  COUNTIF(gender IS NULL)             AS null_gender,
  COUNTIF(marital_status IS NULL)     AS null_marital,
  COUNTIF(language IS NULL)           AS null_language,
  COUNTIF(race IS NULL)               AS null_race,
  COUNTIF(insurance IS NULL)          AS null_insurance,
  COUNTIF(discharge_location IS NULL) AS null_discharge_loc,
  COUNT(DISTINCT gender)             AS nuniq_gender,
  COUNT(DISTINCT marital_status)     AS nuniq_marital,
  COUNT(DISTINCT language)           AS nuniq_language,
  COUNT(DISTINCT race)               AS nuniq_race,
  COUNT(DISTINCT insurance)          AS nuniq_insurance,
  COUNT(DISTINCT discharge_location) AS nuniq_discharge_loc
FROM demo
"""

demo_profile = run_sql(demo_profile_sql)

# Reshape to a readable per-column table.
n = int(demo_profile["n_rows"].iloc[0])
rows = []
for col in ["gender", "marital", "language", "race", "insurance", "discharge_loc"]:
    nulls = int(demo_profile[f"null_{col}"].iloc[0])
    rows.append({
        "column": col,
        "null_rate_pct": round(nulls / n * 100, 1),
        "distinct": int(demo_profile[f"nuniq_{col}"].iloc[0]),
    })
print(f"cohort rows joined: {n:,}  (must equal spine 352,699)")
print(f"distinct hadm_id  : {int(demo_profile['distinct_hadm'].iloc[0]):,}  (must equal rows)\n")
print(pd.DataFrame(rows).to_string(index=False))


cohort rows joined: 352,699  (must equal spine 352,699)
distinct hadm_id  : 352,699  (must equal rows)

       column  null_rate_pct  distinct
       gender            0.0         2
      marital            2.5         4
     language            0.1        25
         race            0.0        33
    insurance            1.1         5
discharge_loc           14.9        13


**Step 2 — Aggregate.** One row per `hadm_id`. Only one real transform here —
`age` derived from MIMIC's anchor scheme; everything else passes through raw
(categorical encoding + the `discharge_location` 14.9% nulls are handled later
in `features_clean`, not in this group). Capped at 90 because MIMIC sets every
patient over 89 to a shifted anchor — ages above 90 aren't real.

This SQL backs `features/feat_demographics.sqlx`, reading `${ref("cohort_split")}`
instead of rebuilding the spine.


In [7]:
# Step 2 — Aggregate. Build the demographics feature row (one per hadm_id).
# Reuse BASE_CTE + spine gates, join patients/admissions, derive age, pass the
# categoricals through raw. In Dataform this reads FROM ${ref("cohort_split")}.
feat_demographics_sql = BASE_CTE + f""",
spine AS (
  SELECT subject_id, hadm_id, admittime
  FROM flagged
  WHERE anchor_age >= 18
    AND hospital_expire_flag = 0 AND deathtime IS NULL
    AND los_hours >= 24
    AND admission_type NOT IN ({_planned_list})
    AND NOT is_continuation
)
SELECT
  s.hadm_id,
  -- age at admission from MIMIC's anchor scheme, capped at 90 (89+ are shifted)
  LEAST(
    p.anchor_age + (EXTRACT(YEAR FROM s.admittime) - p.anchor_year),
    90
  ) AS age,
  p.gender,
  a.marital_status,
  a.language,
  a.race,
  a.admission_type,
  a.insurance,
  a.discharge_location
FROM spine AS s
JOIN `{MIMIC_HOSP}.patients`   AS p USING (subject_id)
JOIN `{MIMIC_HOSP}.admissions` AS a USING (hadm_id)
"""

feat_demographics = run_sql(feat_demographics_sql)
print(f"feat_demographics rows: {len(feat_demographics):,}")
feat_demographics.head()


feat_demographics rows: 352,699


,hadm_id,age,gender,marital_status,language,race,admission_type,insurance,discharge_location
0,22034413,21,M,SINGLE,English,WHITE,EW EMER.,Private,HOME
1,25987122,19,F,SINGLE,English,WHITE,EW EMER.,Private,HOME
2,26209212,19,M,SINGLE,English,WHITE,EW EMER.,Private,HOME
3,21383007,21,F,SINGLE,English,WHITE,EU OBSERVATION,Private,None
4,27502151,21,M,SINGLE,English,WHITE,DIRECT EMER.,None,HOME


In [8]:
# Step 3 — Sanity check. Grain integrity + plausibility.
df = feat_demographics

# (a) one row per hadm_id, matching the spine
assert len(df) == 352_699, f"row count {len(df)} != spine 352,699"
assert df.hadm_id.is_unique, "hadm_id is not unique — join fanned out"

# (b) age within the cohort/anchor bounds (adult gate -> >=18, cap -> <=90)
age_min, age_max = int(df.age.min()), int(df.age.max())
assert 18 <= age_min and age_max <= 90, f"age out of range: [{age_min}, {age_max}]"

print(f"rows            : {len(df):,}  (unique hadm_id: {df.hadm_id.is_unique})")
print(f"age range       : [{age_min}, {age_max}]  median {int(df.age.median())}")
print("\nnull rate by column (should match Step 1 locate):")
print((df.isnull().mean() * 100).round(1).to_string())


rows            : 352,699  (unique hadm_id: True)
age range       : [18, 90]  median 63

null rate by column (should match Step 1 locate):
hadm_id                0.0
age                    0.0
gender                 0.0
marital_status         2.5
language               0.1
race                   0.0
admission_type         0.0
insurance              1.1
discharge_location    14.9


### 3b. Historical Utilization

Past healthcare use — the strongest single signal of chronic illness and
recurring risk. Unlike demographics, these are **true aggregations** over a
patient's history, which makes **leakage the central concern**: every "prior"
metric must look **strictly before the index admission's `admittime`** — never
the index admission itself, never anything after it.

| Feature | Source | Definition |
|---|---|---|
| `prior_admission_count` | `admissions` | # of the patient's earlier admissions with `admittime` < index `admittime`. |
| `prior_inpatient_days` | `admissions` | Σ length-of-stay (days) of those earlier admissions. |
| `recent_ed_visits` | `edstays` | # of ED stays in the `ED_LOOKBACK_DAYS` window before index `admittime`. |
| `index_los_days` | `admissions` | LOS of the index admission itself (known at discharge — not leakage). |

Two design decisions to lock before writing SQL:

- **"Prior" counts over the *full* admission history**, not just cohort
  admissions — a patient's earlier planned/short stays still signal utilization
  even though they're excluded from the cohort spine. So the history side joins
  raw `admissions`, not the spine.
- **`index_los_days` is allowed** — it's measured over the index stay, which is
  complete at the discharge prediction point (forecast origin = discharge).

**Step 1 — Locate.** Before aggregating, confirm: (a) `edstays` coverage — how
many cohort patients have any ED record, and the column we key/time on; (b) the
shape of per-patient admission history (how many have ≥1 prior admission). This
tells us expected null/zero rates so the sanity check has something to compare to.


In [9]:
# Step 1 — Locate. Two probes before aggregating:
#  (a) edstays coverage + the timing column we'll use (intime).
#  (b) admission-history shape: how many cohort admissions have >=1 PRIOR
#      admission (admittime strictly before the index).
ED_LOOKBACK_DAYS = 180  # window for "recent" ED visits before index admittime

# (a) edstays schema peek + coverage against the cohort's patients.
ed_probe_sql = BASE_CTE + f""",
spine AS (
  SELECT subject_id, hadm_id, admittime
  FROM flagged
  WHERE anchor_age >= 18
    AND hospital_expire_flag = 0 AND deathtime IS NULL
    AND los_hours >= 24
    AND admission_type NOT IN ({_planned_list})
    AND NOT is_continuation
)
SELECT
  (SELECT COUNT(*) FROM `{MIMIC_ED}.edstays`)                              AS edstays_total,
  (SELECT COUNT(DISTINCT subject_id) FROM `{MIMIC_ED}.edstays`)           AS edstays_patients,
  COUNT(DISTINCT s.subject_id)                                            AS cohort_patients,
  COUNT(DISTINCT IF(e.subject_id IS NOT NULL, s.subject_id, NULL))        AS cohort_patients_with_ed
FROM spine AS s
LEFT JOIN `{MIMIC_ED}.edstays` AS e USING (subject_id)
"""
ed_probe = run_sql(ed_probe_sql)
print("=== (a) edstays coverage ===")
print(ed_probe.T.rename(columns={0: "value"}).to_string())

# (b) prior-admission shape: for each cohort admission, count earlier admissions.
hist_probe_sql = BASE_CTE + f""",
spine AS (
  SELECT subject_id, hadm_id, admittime
  FROM flagged
  WHERE anchor_age >= 18
    AND hospital_expire_flag = 0 AND deathtime IS NULL
    AND los_hours >= 24
    AND admission_type NOT IN ({_planned_list})
    AND NOT is_continuation
),
prior AS (
  SELECT
    s.hadm_id,
    COUNT(a.hadm_id) AS prior_admission_count
  FROM spine AS s
  JOIN `{MIMIC_HOSP}.admissions` AS a
    ON a.subject_id = s.subject_id
   AND a.admittime < s.admittime          -- STRICTLY prior (leakage guard)
  GROUP BY s.hadm_id
)
SELECT
  (SELECT COUNT(*) FROM spine)                       AS cohort_admissions,
  COUNT(*)                                           AS admissions_with_prior,
  ROUND(AVG(prior_admission_count), 2)               AS avg_prior_when_any,
  MAX(prior_admission_count)                         AS max_prior
FROM prior
"""
hist_probe = run_sql(hist_probe_sql)
print("\n=== (b) prior-admission shape ===")
print(hist_probe.T.rename(columns={0: "value"}).to_string())


=== (a) edstays coverage ===
                          value
edstays_total            425087
edstays_patients         205504
cohort_patients          164847
cohort_patients_with_ed   96414

=== (b) prior-admission shape ===
                        value
cohort_admissions      352699
admissions_with_prior  212258
avg_prior_when_any       5.57
max_prior                 227


**Step 2 — Aggregate.** Four features per `hadm_id`. Absence of history is a
**0, not a null** (locate confirmed ~40% first-admissions, ~42% no-ED), so every
history join is a `LEFT JOIN` + `COALESCE(..., 0)`. The leakage guard lives in
the join predicate: prior admissions key on `a.admittime < s.admittime`
(strictly before the index), and recent ED visits on the
`[index − ED_LOOKBACK_DAYS, index)` window. `index_los_days` is the one metric
about the index stay itself — allowed, since LOS is known at the discharge
forecast origin.

**Data-quality note (caught by Step 3):** the history side reads *raw*
`admissions`, which — unlike the LOS-gated spine — contains a few records with
`dischtime < admittime`. Those would make `prior_inpatient_days` negative, so
each prior admission's LOS is floored at 0 with `GREATEST(..., 0)`: a malformed
row contributes 0 days, never a negative.

This SQL backs `features/feat_utilization.sqlx`, reading `${ref("cohort_split")}`
for the spine but raw `${ref("admissions")}` / `${ref("edstays")}` for history
(full history counts, not just cohort admissions).


In [10]:
# Step 2 — Aggregate. Historical utilization, one row per hadm_id.
# History side joins RAW admissions/edstays (full history, not just cohort).
# All history metrics LEFT JOIN + COALESCE to 0 (absence = a real 0, not null).
feat_utilization_sql = BASE_CTE + f""",
spine AS (
  SELECT subject_id, hadm_id, admittime, dischtime
  FROM flagged
  WHERE anchor_age >= 18
    AND hospital_expire_flag = 0 AND deathtime IS NULL
    AND los_hours >= 24
    AND admission_type NOT IN ({_planned_list})
    AND NOT is_continuation
),
prior_adm AS (
  -- Earlier admissions only: admittime STRICTLY before the index (leakage guard).
  -- Per-row LOS floored at 0: raw admissions (unlike the spine) aren't LOS-gated,
  -- and a few MIMIC records have dischtime < admittime — a malformed row must
  -- contribute 0 days, never a negative.
  SELECT
    s.hadm_id,
    COUNT(a.hadm_id) AS prior_admission_count,
    COALESCE(SUM(GREATEST(TIMESTAMP_DIFF(a.dischtime, a.admittime, HOUR), 0)) / 24.0, 0) AS prior_inpatient_days
  FROM spine AS s
  JOIN `{MIMIC_HOSP}.admissions` AS a
    ON a.subject_id = s.subject_id
   AND a.admittime < s.admittime
  GROUP BY s.hadm_id
),
recent_ed AS (
  -- ED stays starting within [index - ED_LOOKBACK_DAYS, index).
  SELECT
    s.hadm_id,
    COUNT(e.stay_id) AS recent_ed_visits
  FROM spine AS s
  JOIN `{MIMIC_ED}.edstays` AS e
    ON e.subject_id = s.subject_id
   AND e.intime <  s.admittime
   AND e.intime >= TIMESTAMP_SUB(s.admittime, INTERVAL {ED_LOOKBACK_DAYS} DAY)
  GROUP BY s.hadm_id
)
SELECT
  s.hadm_id,
  COALESCE(pa.prior_admission_count, 0)                         AS prior_admission_count,
  COALESCE(pa.prior_inpatient_days, 0)                          AS prior_inpatient_days,
  COALESCE(re.recent_ed_visits, 0)                              AS recent_ed_visits,
  TIMESTAMP_DIFF(s.dischtime, s.admittime, HOUR) / 24.0         AS index_los_days
FROM spine AS s
LEFT JOIN prior_adm AS pa USING (hadm_id)
LEFT JOIN recent_ed AS re USING (hadm_id)
"""

feat_utilization = run_sql(feat_utilization_sql)
print(f"feat_utilization rows: {len(feat_utilization):,}")
feat_utilization.head()


feat_utilization rows: 352,699


,hadm_id,prior_admission_count,prior_inpatient_days,recent_ed_visits,index_los_days
0,28664981,17,26.708333,10,1.958333
1,24962904,18,28.666667,11,1.916667
2,24746267,20,31.458333,12,2.750000
3,24887339,16,71.625000,0,3.833333
4,28184945,17,75.458333,0,1.916667


In [11]:
# Step 3 — Sanity check. Grain integrity + plausibility + no-null guard.
u = feat_utilization

# (a) one row per hadm_id, matching the spine (history CTEs must not fan out)
assert len(u) == 352_699, f"row count {len(u)} != spine 352,699"
assert u.hadm_id.is_unique, "hadm_id is not unique — a history join fanned out"

# (b) no nulls anywhere — every history metric COALESCEd to 0
assert u.isnull().sum().sum() == 0, "unexpected nulls (COALESCE missed a column)"

# (c) plausibility: counts/days non-negative, index LOS >= 1 day (the cohort gate)
assert (u.prior_admission_count >= 0).all(), "negative prior_admission_count"
assert (u.prior_inpatient_days  >= 0).all(), "negative prior_inpatient_days"
assert (u.recent_ed_visits      >= 0).all(), "negative recent_ed_visits"
assert (u.index_los_days >= 1.0).all(), "index_los_days < 1 violates the LOS gate"

print(f"rows                 : {len(u):,}  (unique hadm_id: {u.hadm_id.is_unique})")
print(f"nulls                : {int(u.isnull().sum().sum())}\n")

# Distribution snapshot — should echo the locate probes (~40% zero priors, ~42% no ED).
print("zero-history shares (sanity vs locate):")
print(f"  prior_admission_count == 0 : {(u.prior_admission_count == 0).mean() * 100:5.1f}%  (locate: ~40%)")
print(f"  recent_ed_visits      == 0 : {(u.recent_ed_visits == 0).mean() * 100:5.1f}%")
print("\nnumeric summary:")
print(u[["prior_admission_count", "prior_inpatient_days",
         "recent_ed_visits", "index_los_days"]].describe().round(2).to_string())


rows                 : 352,699  (unique hadm_id: True)
nulls                : 0

zero-history shares (sanity vs locate):
  prior_admission_count == 0 :  39.8%  (locate: ~40%)
  recent_ed_visits      == 0 :  55.3%

numeric summary:
       prior_admission_count  prior_inpatient_days  recent_ed_visits  index_los_days
count               352699.0             352699.00          352699.0       352699.00
mean                    3.35                 16.24              0.82            5.75
std                     6.88                 36.49              1.49            7.37
min                      0.0                  0.00               0.0            1.00
25%                      0.0                  0.00               0.0            2.04
50%                      1.0                  2.92               0.0            3.71
75%                      4.0                 16.04               1.0            6.62
max                    227.0                842.00              67.0          515.54


### 3c. Structured Clinical Codes

The billed **diagnoses and procedures** for the index admission — a compact,
structured summary of *what was actually wrong* and *what was done about it*.
Unlike utilization, **leakage isn't the concern here**: ICD codes are assigned
*for the index stay* and are complete at the discharge forecast origin, so we
read them straight off the index `hadm_id` (no "strictly prior" window).

Both source tables are **long** — many rows per admission, one per coded
condition/procedure (`seq_num` orders them, `seq_num = 1` is primary). The
aggregation collapses that to **one row per `hadm_id`**.

| Feature | Source | Definition |
|---|---|---|
| `diagnosis_count` | `diagnoses_icd` | # of distinct diagnosis codes on the index admission (comorbidity load). |
| `procedure_count` | `procedures_icd` | # of distinct procedure codes on the index admission (intervention intensity). |
| `has_procedure` | `procedures_icd` | whether any procedure was billed (many admissions have none). |

A wrinkle to confirm in locate: MIMIC-IV mixes **ICD-9 and ICD-10** codes
(an `icd_version` column), so the *raw code strings aren't comparable across
versions*. For these **count-based** features that's fine — we're counting
codes, not interpreting them — but it rules out naive per-code flags until we
map to a common vocabulary (deferred; not this group).

**Step 1 — Locate.** Three probes before aggregating: (a) coverage — how many
cohort admissions have ≥1 diagnosis / ≥1 procedure (procedures are often
absent → expect a real 0, not null); (b) the ICD-9/10 version mix in each
table; (c) codes-per-admission shape (avg/max distinct) so the sanity check has
expected ranges. All keyed on the index `hadm_id`.


In [12]:
# Step 1 — Locate. Three probes on the codes tables, keyed on the index hadm_id
# (no "prior" window — ICD codes belong to the index stay and are complete at
# the discharge forecast origin). Reuse BASE_CTE + spine gates for the cohort.
codes_probe_sql = BASE_CTE + f""",
spine AS (
  SELECT hadm_id
  FROM flagged
  WHERE anchor_age >= 18
    AND hospital_expire_flag = 0 AND deathtime IS NULL
    AND los_hours >= 24
    AND admission_type NOT IN ({_planned_list})
    AND NOT is_continuation
),
dx AS (
  SELECT s.hadm_id, COUNT(DISTINCT d.icd_code) AS n_dx
  FROM spine AS s
  JOIN `{MIMIC_HOSP}.diagnoses_icd` AS d USING (hadm_id)
  GROUP BY s.hadm_id
),
px AS (
  SELECT s.hadm_id, COUNT(DISTINCT p.icd_code) AS n_px
  FROM spine AS s
  JOIN `{MIMIC_HOSP}.procedures_icd` AS p USING (hadm_id)
  GROUP BY s.hadm_id
)
SELECT
  (SELECT COUNT(*) FROM spine)                  AS cohort_admissions,
  -- (a) coverage: how many cohort admissions carry >=1 code in each table
  (SELECT COUNT(*) FROM dx)                      AS adm_with_diagnosis,
  (SELECT COUNT(*) FROM px)                      AS adm_with_procedure,
  -- (c) codes-per-admission shape (only over admissions that have any)
  (SELECT ROUND(AVG(n_dx), 2) FROM dx)          AS avg_dx_when_any,
  (SELECT MAX(n_dx) FROM dx)                     AS max_dx,
  (SELECT ROUND(AVG(n_px), 2) FROM px)          AS avg_px_when_any,
  (SELECT MAX(n_px) FROM px)                     AS max_px
"""
codes_probe = run_sql(codes_probe_sql)
n_cohort = int(codes_probe["cohort_admissions"].iloc[0])
print("=== (a) coverage + (c) codes-per-admission shape ===")
prof = codes_probe.T.rename(columns={0: "value"})
print(prof.to_string())
print(f"\n  diagnosis coverage: {int(codes_probe['adm_with_diagnosis'].iloc[0]) / n_cohort * 100:5.1f}%")
print(f"  procedure coverage: {int(codes_probe['adm_with_procedure'].iloc[0]) / n_cohort * 100:5.1f}%"
      "   (expect well under 100% — many stays have no billed procedure)")

# (b) ICD-9/10 version mix in each table (cohort-restricted), since raw code
# strings aren't comparable across versions — confirms count-only features.
version_sql = BASE_CTE + f""",
spine AS (
  SELECT hadm_id
  FROM flagged
  WHERE anchor_age >= 18
    AND hospital_expire_flag = 0 AND deathtime IS NULL
    AND los_hours >= 24
    AND admission_type NOT IN ({_planned_list})
    AND NOT is_continuation
)
SELECT 'diagnoses'  AS tbl, d.icd_version AS icd_version, COUNT(*) AS n
FROM spine AS s JOIN `{MIMIC_HOSP}.diagnoses_icd`  AS d USING (hadm_id)
GROUP BY icd_version
UNION ALL
SELECT 'procedures' AS tbl, p.icd_version AS icd_version, COUNT(*) AS n
FROM spine AS s JOIN `{MIMIC_HOSP}.procedures_icd` AS p USING (hadm_id)
GROUP BY icd_version
ORDER BY tbl, icd_version
"""
version_mix = run_sql(version_sql)
print("\n=== (b) ICD version mix (cohort) ===")
print(version_mix.to_string(index=False))


=== (a) coverage + (c) codes-per-admission shape ===
                     value
cohort_admissions   352699
adm_with_diagnosis  352496
adm_with_procedure  199167
avg_dx_when_any      13.34
max_dx                  56
avg_px_when_any       2.76
max_px                  37

  diagnosis coverage:  99.9%
  procedure coverage:  56.5%   (expect well under 100% — many stays have no billed procedure)

=== (b) ICD version mix (cohort) ===
       tbl  icd_version       n
 diagnoses            9 2115591
 diagnoses           10 2587555
procedures            9  304328
procedures           10  276972


**Step 2 — Aggregate.** Three features per `hadm_id`. Locate confirmed
diagnoses ≈ 99.9% coverage but procedures only ~56.5%, so absence is a **real 0,
not a null** — both code tables `LEFT JOIN` + `COUNT DISTINCT icd_code` collapse
the long tables to one row, with `COALESCE(..., 0)` for the ~43.5% of stays with
no procedure (and the handful with no diagnosis). `has_procedure` is just
`procedure_count > 0`, surfaced as an explicit flag because "had any procedure"
is a cleaner signal than the count alone for the many zero-procedure stays.

`COUNT DISTINCT icd_code` (not `COUNT(*)`) mirrors the locate definition and is
safe across the ICD-9/10 mix — we're counting *how many distinct conditions /
procedures*, never comparing the code strings themselves.

This SQL backs `features/feat_codes.sqlx`, reading `${ref("cohort_split")}` for
the spine and raw `${ref("diagnoses_icd")}` / `${ref("procedures_icd")}` for the
codes.


In [13]:
# Step 2 — Aggregate. Structured clinical codes, one row per hadm_id.
# Both code tables LEFT JOIN + COUNT DISTINCT icd_code (matches locate), then
# COALESCE to 0 — absence of a procedure (~43.5% of stays) is a real 0, not null.
feat_codes_sql = BASE_CTE + f""",
spine AS (
  SELECT hadm_id
  FROM flagged
  WHERE anchor_age >= 18
    AND hospital_expire_flag = 0 AND deathtime IS NULL
    AND los_hours >= 24
    AND admission_type NOT IN ({_planned_list})
    AND NOT is_continuation
),
dx AS (
  -- distinct diagnosis codes on the index admission (comorbidity load).
  SELECT hadm_id, COUNT(DISTINCT icd_code) AS diagnosis_count
  FROM `{MIMIC_HOSP}.diagnoses_icd`
  GROUP BY hadm_id
),
px AS (
  -- distinct procedure codes on the index admission (intervention intensity).
  SELECT hadm_id, COUNT(DISTINCT icd_code) AS procedure_count
  FROM `{MIMIC_HOSP}.procedures_icd`
  GROUP BY hadm_id
)
SELECT
  s.hadm_id,
  COALESCE(dx.diagnosis_count, 0)       AS diagnosis_count,
  COALESCE(px.procedure_count, 0)       AS procedure_count,
  COALESCE(px.procedure_count, 0) > 0   AS has_procedure
FROM spine AS s
LEFT JOIN dx USING (hadm_id)
LEFT JOIN px USING (hadm_id)
"""

feat_codes = run_sql(feat_codes_sql)
print(f"feat_codes rows: {len(feat_codes):,}")
feat_codes.head()


feat_codes rows: 352,699


,hadm_id,diagnosis_count,procedure_count,has_procedure
0,20399821,39,10,True
1,29101414,19,11,True
2,21227547,39,12,True
3,24239692,30,10,True
4,28634286,31,10,True


In [14]:
# Step 3 — Sanity check. Grain integrity + plausibility + no-null guard.
c = feat_codes

# (a) one row per hadm_id, matching the spine (GROUP BY collapse must be 1:1)
assert len(c) == 352_699, f"row count {len(c)} != spine 352,699"
assert c.hadm_id.is_unique, "hadm_id is not unique — a codes join fanned out"

# (b) no nulls — both counts COALESCEd to 0, has_procedure derived from a count
assert c.isnull().sum().sum() == 0, "unexpected nulls (COALESCE missed a column)"

# (c) plausibility: counts non-negative and within the locate-observed ceilings
assert (c.diagnosis_count >= 0).all(), "negative diagnosis_count"
assert (c.procedure_count >= 0).all(), "negative procedure_count"
assert c.diagnosis_count.max() <= 56, f"diagnosis_count {c.diagnosis_count.max()} exceeds locate max 56"
assert c.procedure_count.max() <= 37, f"procedure_count {c.procedure_count.max()} exceeds locate max 37"

# (d) flag must agree with the count it's derived from
assert (c.has_procedure == (c.procedure_count > 0)).all(), "has_procedure disagrees with procedure_count"

print(f"rows                 : {len(c):,}  (unique hadm_id: {c.hadm_id.is_unique})")
print(f"nulls                : {int(c.isnull().sum().sum())}\n")

# Distribution snapshot — should echo locate (dx ~100% present, px ~56.5%).
print("zero / coverage shares (sanity vs locate):")
print(f"  diagnosis_count == 0 : {(c.diagnosis_count == 0).mean() * 100:5.1f}%  (locate: ~0.1%)")
print(f"  has_procedure        : {c.has_procedure.mean() * 100:5.1f}%  (locate coverage: ~56.5%)")
print("\nnumeric summary:")
print(c[["diagnosis_count", "procedure_count"]].describe().round(2).to_string())


rows                 : 352,699  (unique hadm_id: True)
nulls                : 0

zero / coverage shares (sanity vs locate):
  diagnosis_count == 0 :   0.1%  (locate: ~0.1%)
  has_procedure        :  56.5%  (locate coverage: ~56.5%)

numeric summary:
       diagnosis_count  procedure_count
count         352699.0         352699.0
mean             13.33             1.56
std               7.55             2.29
min                0.0              0.0
25%                8.0              0.0
50%               12.0              1.0
75%               18.0              2.0
max               56.0             37.0


### 3d. Medications

The drugs ordered during the index admission — a proxy for **treatment
intensity and regimen complexity**. Polypharmacy (many concurrent drugs) is a
well-established readmission signal, and a few high-risk drug classes
(anticoagulants, insulin) flag conditions that are fragile to manage post-discharge.

Unlike codes, **leakage *is* a concern here** in a subtle way: `prescriptions`
rows carry `starttime` / `stoptime`, and not every row is necessarily confined
to the index stay — discharge meds and the occasional malformed timestamp can
sit at or past `dischtime`. We key on the index `hadm_id` (which already scopes
to the stay) and additionally **gate `starttime <= dischtime`** so a feature
never reflects an order placed after the discharge forecast origin.

| Feature | Source | Definition |
|---|---|---|
| `medication_count` | `prescriptions` | # of distinct drugs ordered during the index stay (polypharmacy). |
| `medication_order_count` | `prescriptions` | # of prescription rows (order volume; ≥ distinct drugs). |
| `on_anticoagulant` | `prescriptions` | any anticoagulant ordered (bleeding/clotting fragility). |
| `on_insulin` | `prescriptions` | any insulin ordered (diabetic management complexity). |

`medication_count` uses `COUNT(DISTINCT drug)` — but `drug` is **free-text**
(brand/formulation variants), so it's a *coarse* polypharmacy proxy, not a clean
ingredient count. The two class flags use case-insensitive `LIKE` matches on a
small keyword list, kept visible as a Python constant so the vocabulary is auditable.

**Step 1 — Locate.** Four probes before aggregating: (a) coverage — how many
cohort admissions have ≥1 prescription (expect near-universal, but confirm);
(b) the timing columns + how many rows have `starttime > dischtime` (the leak we're
gating); (c) distinct-drug shape per admission (avg/max) for plausibility bounds;
(d) a peek at the `drug` vocabulary to sanity-check the anticoagulant/insulin
keyword matches before committing to them.


In [15]:
# Step 1 — Locate. Four probes on prescriptions, keyed on the index hadm_id.
# Drug-class keyword lists kept visible/auditable; case-insensitive LIKE match.
# NOTE: `heparin` is intentionally EXCLUDED — in MIMIC nearly every inpatient
# gets it as DVT prophylaxis or line/catheter flushes, so it swept ~79% of the
# cohort and the flag lost discriminative value. Keeping warfarin + the DOACs +
# enoxaparin targets therapeutic anticoagulation more cleanly.
ANTICOAGULANT_KEYWORDS = ["warfarin", "enoxaparin", "apixaban",
                          "rivaroxaban", "dabigatran", "fondaparinux"]
INSULIN_KEYWORDS = ["insulin"]

_anticoag_like = " OR ".join(f"LOWER(drug) LIKE '%{k}%'" for k in ANTICOAGULANT_KEYWORDS)
_insulin_like  = " OR ".join(f"LOWER(drug) LIKE '%{k}%'" for k in INSULIN_KEYWORDS)

# (a) coverage + (b) timing leak + (c) distinct-drug shape, all in one pass.
rx_probe_sql = BASE_CTE + f""",
spine AS (
  SELECT hadm_id, dischtime
  FROM flagged
  WHERE anchor_age >= 18
    AND hospital_expire_flag = 0 AND deathtime IS NULL
    AND los_hours >= 24
    AND admission_type NOT IN ({_planned_list})
    AND NOT is_continuation
),
rx AS (
  SELECT
    s.hadm_id,
    COUNT(*)                                              AS n_orders,
    COUNT(DISTINCT r.drug)                                AS n_drugs,
    COUNTIF(r.starttime > s.dischtime)                    AS n_after_disch,
    COUNTIF(r.starttime IS NULL)                          AS n_null_start
  FROM spine AS s
  JOIN `{MIMIC_HOSP}.prescriptions` AS r USING (hadm_id)
  GROUP BY s.hadm_id
)
SELECT
  (SELECT COUNT(*) FROM spine)              AS cohort_admissions,
  COUNT(*)                                  AS adm_with_rx,
  ROUND(AVG(n_drugs), 2)                    AS avg_drugs_when_any,
  MAX(n_drugs)                              AS max_drugs,
  ROUND(AVG(n_orders), 2)                   AS avg_orders_when_any,
  MAX(n_orders)                             AS max_orders,
  SUM(n_after_disch)                        AS rows_started_after_disch,
  SUM(n_null_start)                         AS rows_null_starttime
FROM rx
"""
rx_probe = run_sql(rx_probe_sql)
n_cohort = int(rx_probe["cohort_admissions"].iloc[0])
print("=== (a) coverage + (b) timing leak + (c) drug-count shape ===")
print(rx_probe.T.rename(columns={0: "value"}).to_string())
print(f"\n  prescription coverage: {int(rx_probe['adm_with_rx'].iloc[0]) / n_cohort * 100:5.1f}%")
print(f"  rows started AFTER dischtime (the leak we gate): {int(rx_probe['rows_started_after_disch'].iloc[0]):,}")
print(f"  rows with NULL starttime: {int(rx_probe['rows_null_starttime'].iloc[0]):,}")

# (d) drug-class keyword sanity: cohort-level share of admissions matching each
# class, so we can confirm the keyword lists hit plausible prevalences before
# committing. Gated to in-stay orders (starttime <= dischtime).
rx_class_sql = BASE_CTE + f""",
spine AS (
  SELECT hadm_id, dischtime
  FROM flagged
  WHERE anchor_age >= 18
    AND hospital_expire_flag = 0 AND deathtime IS NULL
    AND los_hours >= 24
    AND admission_type NOT IN ({_planned_list})
    AND NOT is_continuation
),
flags AS (
  SELECT
    s.hadm_id,
    MAX(CASE WHEN {_anticoag_like} THEN 1 ELSE 0 END) AS on_anticoag,
    MAX(CASE WHEN {_insulin_like}  THEN 1 ELSE 0 END) AS on_insulin
  FROM spine AS s
  JOIN `{MIMIC_HOSP}.prescriptions` AS r
    ON r.hadm_id = s.hadm_id
   AND (r.starttime IS NULL OR r.starttime <= s.dischtime)   -- in-stay gate
  GROUP BY s.hadm_id
)
SELECT
  (SELECT COUNT(*) FROM spine)  AS cohort_admissions,
  SUM(on_anticoag)              AS adm_on_anticoag,
  SUM(on_insulin)               AS adm_on_insulin
FROM flags
"""
rx_class = run_sql(rx_class_sql)
nc = int(rx_class["cohort_admissions"].iloc[0])
print("\n=== (d) drug-class keyword prevalence (in-stay gated) ===")
print(f"  on_anticoagulant: {int(rx_class['adm_on_anticoag'].iloc[0]) / nc * 100:5.1f}%  "
      f"(keywords: {', '.join(ANTICOAGULANT_KEYWORDS)})")
print(f"  on_insulin      : {int(rx_class['adm_on_insulin'].iloc[0]) / nc * 100:5.1f}%  "
      f"(keywords: {', '.join(INSULIN_KEYWORDS)})")


=== (a) coverage + (b) timing leak + (c) drug-count shape ===
                           value
cohort_admissions         352699
adm_with_rx               341601
avg_drugs_when_any         24.56
max_drugs                    196
avg_orders_when_any        45.16
max_orders                  2812
rows_started_after_disch  119424
rows_null_starttime        17728

  prescription coverage:  96.9%
  rows started AFTER dischtime (the leak we gate): 119,424
  rows with NULL starttime: 17,728

=== (d) drug-class keyword prevalence (in-stay gated) ===
  on_anticoagulant:  25.4%  (keywords: warfarin, enoxaparin, apixaban, rivaroxaban, dabigatran, fondaparinux)
  on_insulin      :  32.0%  (keywords: insulin)


In [16]:
# Step 1 (cont.) — Plausibility of the count outliers. avg_orders 45,
# max_orders 2,812 and max_drugs 196 look extreme; before treating them as
# corruption, test the two things that would make them *real*:
#   1. the tail is smooth (max is the top of a continuum, not a lone spike), and
#   2. the counts track length-of-stay (a multi-week ICU course legitimately
#      re-orders dozens of drips/titrations daily -> thousands of orders).
# If both hold, the outliers are sick-patient signal, not bad data, and we keep
# the raw counts (no capping) — the sanity-check ceilings just come from here.
rx_outlier_sql = BASE_CTE + f""",
spine AS (
  SELECT hadm_id, dischtime,
         TIMESTAMP_DIFF(dischtime, admittime, HOUR) / 24.0 AS los_days
  FROM flagged
  WHERE anchor_age >= 18
    AND hospital_expire_flag = 0 AND deathtime IS NULL
    AND los_hours >= 24
    AND admission_type NOT IN ({_planned_list})
    AND NOT is_continuation
),
rx AS (
  SELECT
    s.hadm_id,
    s.los_days,
    COUNT(*)               AS n_orders,
    COUNT(DISTINCT r.drug) AS n_drugs
  FROM spine AS s
  JOIN `{MIMIC_HOSP}.prescriptions` AS r
    ON r.hadm_id = s.hadm_id
   AND (r.starttime IS NULL OR r.starttime <= s.dischtime)   -- in-stay gate
  GROUP BY s.hadm_id, s.los_days
)
SELECT
  APPROX_QUANTILES(n_drugs, 100)[OFFSET(50)]  AS drugs_p50,
  APPROX_QUANTILES(n_drugs, 100)[OFFSET(90)]  AS drugs_p90,
  APPROX_QUANTILES(n_drugs, 100)[OFFSET(99)]  AS drugs_p99,
  MAX(n_drugs)                                AS drugs_max,
  APPROX_QUANTILES(n_orders, 100)[OFFSET(50)] AS orders_p50,
  APPROX_QUANTILES(n_orders, 100)[OFFSET(90)] AS orders_p90,
  APPROX_QUANTILES(n_orders, 100)[OFFSET(99)] AS orders_p99,
  MAX(n_orders)                               AS orders_max,
  ROUND(CORR(n_orders, los_days), 3)          AS corr_orders_los,
  ROUND(CORR(n_drugs,  los_days), 3)          AS corr_drugs_los
FROM rx
"""
rx_outlier = run_sql(rx_outlier_sql)
print("=== count-tail percentiles (in-stay gated) ===")
print(rx_outlier.T.rename(columns={0: "value"}).to_string())
print(f"\n  corr(n_orders, los_days): {rx_outlier['corr_orders_los'].iloc[0]:+.3f}"
      "   (strong + => long stays drive the order volume)")
print(f"  corr(n_drugs,  los_days): {rx_outlier['corr_drugs_los'].iloc[0]:+.3f}")

# Peek at the top admissions by order volume — confirm each extreme is a long
# stay (plausible), not a 1-day admission with thousands of orders (suspect).
rx_top_sql = BASE_CTE + f""",
spine AS (
  SELECT hadm_id, dischtime,
         TIMESTAMP_DIFF(dischtime, admittime, HOUR) / 24.0 AS los_days
  FROM flagged
  WHERE anchor_age >= 18
    AND hospital_expire_flag = 0 AND deathtime IS NULL
    AND los_hours >= 24
    AND admission_type NOT IN ({_planned_list})
    AND NOT is_continuation
)
SELECT
  s.hadm_id,
  ROUND(s.los_days, 1)   AS los_days,
  COUNT(*)               AS n_orders,
  COUNT(DISTINCT r.drug) AS n_drugs,
  ROUND(COUNT(*) / s.los_days, 1) AS orders_per_day
FROM spine AS s
JOIN `{MIMIC_HOSP}.prescriptions` AS r
  ON r.hadm_id = s.hadm_id
 AND (r.starttime IS NULL OR r.starttime <= s.dischtime)
GROUP BY s.hadm_id, s.los_days
ORDER BY n_orders DESC
LIMIT 10
"""
rx_top = run_sql(rx_top_sql)
print("\n=== top 10 admissions by order volume ===")
print(rx_top.to_string(index=False))


=== count-tail percentiles (in-stay gated) ===
                 value
drugs_p50           21
drugs_p90           42
drugs_p99           75
drugs_max          196
orders_p50          29
orders_p90          91
orders_p99         255
orders_max        2811
corr_orders_los  0.812
corr_drugs_los   0.703

  corr(n_orders, los_days): +0.812   (strong + => long stays drive the order volume)
  corr(n_drugs,  los_days): +0.703

=== top 10 admissions by order volume ===
 hadm_id  los_days  n_orders  n_drugs  orders_per_day
26415640     515.5      2811      145             5.5
24673862     238.3      2169      183             9.1
29642964     219.9      1785      193             8.1
29850931     226.5      1766      181             7.8
24605316     296.0      1698      196             5.7
24784126     309.0      1580      151             5.1
23761571     151.1      1516      173            10.0
26571961     181.7      1435      176             7.9
25741205     216.0      1382      141             

**Step 2 — Aggregate.** Four features per `hadm_id`. Locate confirmed 96.9%
coverage, so the 3.1% of stays with no prescription get a **real 0, not a null**
— `prescriptions` `LEFT JOIN` + `COALESCE(..., 0)` on the counts. The plausibility
probe cleared the extreme counts (smooth tails, `corr ≈ 0.8` with LOS), so the
raw `COUNT(DISTINCT drug)` / `COUNT(*)` ship **uncapped**.

Every aggregate sits behind the in-stay leakage gate
`starttime IS NULL OR starttime <= dischtime` — orders placed after the discharge
forecast origin (119,424 rows) never contribute, while the 17,728 NULL-`starttime`
rows are kept as in-stay. The two class flags collapse to one row with
`MAX(CASE WHEN <keyword LIKE> THEN 1 ELSE 0 END)` (anything ordered once → 1).

`medication_count` stays a **coarse** polypharmacy proxy: `drug` is free-text, so
brand/formulation variants inflate the distinct count — fine as a relative signal,
not a clean ingredient tally.

This SQL backs `features/feat_medications.sqlx`, reading `${ref("cohort_split")}`
for the spine and raw `${ref("prescriptions")}` for the orders.


In [17]:
# Step 2 — Aggregate. Medications, one row per hadm_id. prescriptions LEFT JOIN
# + COALESCE to 0 (3.1% of stays have no rx -> a real 0). In-stay leakage gate
# on every aggregate: starttime IS NULL OR starttime <= dischtime. Class flags
# via MAX(CASE ...) on the visible keyword lists. In Dataform the spine reads
# FROM ${ref("cohort_split")} and the orders FROM ${ref("prescriptions")}.
feat_medications_sql = BASE_CTE + f""",
spine AS (
  SELECT hadm_id, dischtime
  FROM flagged
  WHERE anchor_age >= 18
    AND hospital_expire_flag = 0 AND deathtime IS NULL
    AND los_hours >= 24
    AND admission_type NOT IN ({_planned_list})
    AND NOT is_continuation
),
rx AS (
  -- In-stay orders only (gate drops the 119k rows placed after dischtime).
  -- COUNT(DISTINCT drug) is a coarse polypharmacy proxy (drug is free-text).
  SELECT
    s.hadm_id,
    COUNT(DISTINCT r.drug)                              AS medication_count,
    COUNT(*)                                            AS medication_order_count,
    MAX(CASE WHEN {_anticoag_like} THEN 1 ELSE 0 END)   AS on_anticoagulant,
    MAX(CASE WHEN {_insulin_like}  THEN 1 ELSE 0 END)   AS on_insulin
  FROM spine AS s
  JOIN `{MIMIC_HOSP}.prescriptions` AS r
    ON r.hadm_id = s.hadm_id
   AND (r.starttime IS NULL OR r.starttime <= s.dischtime)
  GROUP BY s.hadm_id
)
SELECT
  s.hadm_id,
  COALESCE(rx.medication_count, 0)                      AS medication_count,
  COALESCE(rx.medication_order_count, 0)                AS medication_order_count,
  COALESCE(rx.on_anticoagulant, 0) = 1                  AS on_anticoagulant,
  COALESCE(rx.on_insulin, 0) = 1                        AS on_insulin
FROM spine AS s
LEFT JOIN rx USING (hadm_id)
"""

feat_medications = run_sql(feat_medications_sql)
print(f"feat_medications rows: {len(feat_medications):,}")
feat_medications.head()


feat_medications rows: 352,699


,hadm_id,medication_count,medication_order_count,on_anticoagulant,on_insulin
0,28258130,94,314,True,True
1,27259207,50,83,False,True
2,27996267,81,187,False,True
3,27920583,49,99,False,True
4,29023346,100,432,False,True


In [18]:
# Step 3 — Sanity check. Grain integrity + plausibility + no-null guard.
m = feat_medications

# (a) one row per hadm_id, matching the spine (the rx GROUP BY must stay 1:1)
assert len(m) == 352_699, f"row count {len(m)} != spine 352,699"
assert m.hadm_id.is_unique, "hadm_id is not unique — the rx join fanned out"

# (b) no nulls — counts COALESCEd to 0, flags COALESCEd to a boolean
assert m.isnull().sum().sum() == 0, "unexpected nulls (COALESCE missed a column)"

# (c) plausibility: counts non-negative and within the locate-observed ceilings
#     (the plausibility probe cleared these as real long-stay signal, uncapped).
assert (m.medication_count >= 0).all(), "negative medication_count"
assert (m.medication_order_count >= 0).all(), "negative medication_order_count"
assert m.medication_count.max() <= 196, f"medication_count {m.medication_count.max()} exceeds locate max 196"
assert m.medication_order_count.max() <= 2811, f"medication_order_count {m.medication_order_count.max()} exceeds locate max 2811"

# (d) order_count >= distinct drug_count by construction (rows >= distinct values)
assert (m.medication_order_count >= m.medication_count).all(), "order_count < distinct drug_count"

print(f"rows                 : {len(m):,}  (unique hadm_id: {m.hadm_id.is_unique})")
print(f"nulls                : {int(m.isnull().sum().sum())}\n")

# Distribution snapshot — should echo locate (3.1% no-rx zeros, flags ~25%/32%).
print("zero / flag shares (sanity vs locate):")
print(f"  medication_count == 0 : {(m.medication_count == 0).mean() * 100:5.1f}%  (locate no-rx: ~3.1%)")
print(f"  on_anticoagulant      : {m.on_anticoagulant.mean() * 100:5.1f}%  (locate: ~25.4%)")
print(f"  on_insulin            : {m.on_insulin.mean() * 100:5.1f}%  (locate: ~32.0%)")
print("\nnumeric summary:")
print(m[["medication_count", "medication_order_count"]].describe().round(2).to_string())


rows                 : 352,699  (unique hadm_id: True)
nulls                : 0

zero / flag shares (sanity vs locate):
  medication_count == 0 :   3.1%  (locate no-rx: ~3.1%)
  on_anticoagulant      :  25.4%  (locate: ~25.4%)
  on_insulin            :  32.0%  (locate: ~32.0%)

numeric summary:
       medication_count  medication_order_count
count          352699.0                352699.0
mean              23.67                   43.41
std                14.5                   51.88
min                 0.0                     0.0
25%                14.0                    18.0
50%                21.0                    28.0
75%                29.0                    49.0
max               196.0                  2811.0


### 3e. Physiology & Labs

The patient's **physiological state at the discharge forecast origin** — the
single richest signal for a 30-day bounce-back. Rather than a generic chemistry
panel, we target a **research-grounded analyte set** repeatedly reported as
strong, near-linear readmission predictors in MIMIC-IV work:

| Analyte | Why (literature) |
|---|---|
| **RBC** (red blood cell count) | core blood-health / organ-function marker; near-linear risk |
| **RDW** (red cell distribution width) | strongest of the set; anisocytosis tracks chronic illness; linear |
| **Glucose** | metabolic stress / diabetes control |
| **Monocytes** | systemic inflammation |

Scope is **`labevents` only** this group (hosp-wide labs). Hemodynamics
(heart rate, blood pressure) are *bedside vitals* in `chartevents`, not labs — held
for a separate ICU-physiology discussion so this group stays purely lab-based.

**Leakage** is the central concern, and subtler than medications: each result
carries a `charttime`, and labs are often drawn around — or resulted after —
discharge. Every aggregate is gated to the in-stay window
`charttime <= dischtime` (and `>= admittime`) so no feature reflects a value
observed after the forecast origin.

**Extraction shape — state at discharge + late-stay instability.** Per analyte,
five features (≈20 columns total). The philosophy is "value at discharge and how
it changed," not a generic stay-wide mean:

| Feature | Definition |
|---|---|
| `<analyte>_last` | last value before discharge (the prediction-origin state; where the linear risk acts) |
| `<analyte>_max` | stay maximum (how deranged it got) |
| `<analyte>_min` | stay minimum |
| `<analyte>_delta` | last − first in-stay value (the "sudden fluctuation before going home" instability signal) |
| `<analyte>_measured` | was it drawn at all? (ordering is itself signal; tells `features_clean` where to impute) |

Low-signal columns are pruned later in **feature selection** — this stage is
faithful extraction, not winnowing.

**Step 1 — Locate.** Before any aggregation, resolve two unknowns per analyte
against the cohort: (a) the exact `d_labitems` **`itemid`(s) + unit**
(MIMIC often has several per concept; monocytes split into absolute count vs.
percent), and (b) **coverage** — what fraction of the 352,699 stays have ≥1
in-stay result (drives how much `features_clean` imputes) and ≥2 results
(whether `_delta` is viable). `labevents` is the largest hosp table, so the probe
filters early — restricted to the cohort `hadm_id`s and a candidate `itemid`
shortlist — rather than scanning the whole table.


In [19]:
# Step 1 — Locate. Resolve the itemid(s) + unit per analyte and measure cohort
# coverage BEFORE aggregating. labevents is the largest hosp table, so we prune
# its scan two ways: itemid IN (candidates resolved from the small d_labitems
# dim) and hadm_id IN the cohort (via the spine join). Every result is in-stay
# gated: admittime <= charttime <= dischtime. We don't hardcode itemids — we
# discover them by label keyword and let the per-itemid coverage tell us which
# to keep (MIMIC often has several itemids per concept; monocytes split into
# an absolute count vs a percent, which this probe will surface as separate rows).
LAB_ANALYTES = {
    "rbc":       ["red blood cell"],
    "rdw":       ["red cell distribution width", "rdw"],
    "glucose":   ["glucose"],
    "monocytes": ["monocyte"],
}

# Map each candidate label to its analyte tag (CASE ladder), and the union
# predicate that shortlists candidate itemids from d_labitems.
_analyte_case = "\n".join(
    "      WHEN " + " OR ".join(f"LOWER(label) LIKE '%{k}%'" for k in kws)
    + f" THEN '{tag}'"
    for tag, kws in LAB_ANALYTES.items()
)
_label_like = " OR ".join(
    f"LOWER(label) LIKE '%{k}%'" for kws in LAB_ANALYTES.values() for k in kws
)

labitems_probe_sql = BASE_CTE + f""",
spine AS (
  SELECT hadm_id, admittime, dischtime
  FROM flagged
  WHERE anchor_age >= 18
    AND hospital_expire_flag = 0 AND deathtime IS NULL
    AND los_hours >= 24
    AND admission_type NOT IN ({_planned_list})
    AND NOT is_continuation
),
items AS (
  -- Shortlist candidate lab itemids from the small dimension table, tagging
  -- each with its analyte so we can group coverage by concept.
  SELECT
    itemid, label, fluid, category,
    CASE
{_analyte_case}
    END AS analyte
  FROM `{MIMIC_HOSP}.d_labitems`
  WHERE {_label_like}
),
le AS (
  -- In-stay results for the candidate itemids only (itemid IN items + cohort).
  SELECT
    it.analyte, it.itemid, it.label, it.fluid,
    le.hadm_id, le.valuenum, le.valueuom
  FROM spine AS s
  JOIN `{MIMIC_HOSP}.labevents` AS le
    ON le.hadm_id = s.hadm_id
   AND le.charttime >= s.admittime
   AND le.charttime <= s.dischtime
  JOIN items AS it ON it.itemid = le.itemid
)
SELECT
  analyte,
  itemid,
  label,
  fluid,
  ANY_VALUE(valueuom)                                        AS unit_sample,
  COUNT(*)                                                   AS n_results,
  COUNT(DISTINCT hadm_id)                                    AS n_adm,
  ROUND(COUNT(DISTINCT hadm_id)
        / (SELECT COUNT(*) FROM spine) * 100, 1)             AS coverage_pct,
  ROUND(AVG(valuenum), 2)                                    AS mean_val,
  COUNTIF(valuenum IS NULL)                                  AS n_null_valuenum
FROM le
GROUP BY analyte, itemid, label, fluid
ORDER BY analyte, n_results DESC
"""

labitems_probe = run_sql(labitems_probe_sql)
print("=== candidate lab itemids — coverage on the cohort (in-stay gated) ===")
print("(pick the blood itemid(s) with the dominant coverage per analyte;")
print(" watch monocytes for an absolute-count vs percent split)\n")
print(labitems_probe.to_string(index=False))

# Per-analyte rollup: combined coverage across whichever itemids we'd keep.
print("\n=== per-analyte combined coverage ===")
_rollup = (
    labitems_probe.groupby("analyte")
    .agg(itemids=("itemid", "nunique"),
         results=("n_results", "sum"),
         max_single_coverage=("coverage_pct", "max"))
    .reset_index()
)
print(_rollup.to_string(index=False))


=== candidate lab itemids — coverage on the cohort (in-stay gated) ===
(pick the blood itemid(s) with the dominant coverage per analyte;
 watch monocytes for an absolute-count vs percent split)

  analyte  itemid                   label               fluid unit_sample  n_results  n_adm  coverage_pct  mean_val  n_null_valuenum
  glucose   50931                 Glucose               Blood       mg/dL    1997610 302514          85.8    128.66              229
  glucose   51478                 Glucose               Urine       mg/dL     172582 110360          31.3    374.76           156557
  glucose   50809                 Glucose               Blood       mg/dL     125406  22837           6.5    154.17              227
  glucose   51790            Glucose, CSF Cerebrospinal Fluid       mg/dL       6767   5477           1.6     73.34                8
  glucose   51053        Glucose, Pleural             Pleural       mg/dL       5770   4493           1.3    117.93              104
  gluco

**Step 2 — Aggregate.** Twenty features (5 × 4 analytes) per `hadm_id`. Locate
locked the **blood** itemid per analyte, kept visible as `LAB_ITEMIDS`:

| Analyte | itemid | Unit | Coverage |
|---|---|---|---|
| RBC | `51279` | m/uL | ~89.6% |
| RDW | `51277` | % | ~89.6% |
| Glucose | `50931` | mg/dL (Blood) | ~85.8% |
| Monocytes | `52074` | K/uL (absolute count) | ~16.4% |

The monocyte **absolute count** was chosen over the percent variant for clinical
fidelity (it's the inflammation *count* the literature points to); its lower
coverage is carried cleanly by the `_measured` flag + downstream imputation.

Every result is in-stay gated `admittime <= charttime <= dischtime`, and
`valuenum > 0` — text-only results have no `valuenum`, and a 0/negative reading
is a physiologically impossible junk value for all four analytes (an RBC of 0 is
incompatible with life), so it's dropped rather than allowed to poison `_min` /
`_delta`. Per analyte:

- `_last` / `_first` come from `ARRAY_AGG(... IGNORE NULLS ORDER BY charttime
  DESC/ASC LIMIT 1)` — the most-recent and earliest in-stay values. `_last` is
  the prediction-origin state; `_first` exists only to form `_delta`.
- `_max` / `_min` are `MAX/MIN(IF(analyte=...))` conditional aggregates.
- `_delta = _last − _first` — the late-stay instability signal.
- `_measured = COUNT > 0` — whether the analyte was drawn at all.

**Missing is NULL here, not 0** — the key difference from the count groups
(codes, medications). A lab that was never drawn has *no value*, not a value of
zero, so unmeasured analytes stay `NULL` (no `COALESCE`). Those NULLs flow to
`features_clean`, which imputes them with **train-only** statistics (the leakage
guard) and keeps `_measured` as its own feature. Feature selection later prunes
whatever doesn't earn its place (the sparse monocyte columns are the prime
candidates).

This SQL backs `features/feat_labs.sqlx`, reading `${ref("cohort_split")}` for
the spine and raw `${ref("labevents")}` for the results.


In [23]:
# Step 2 — Aggregate. Physiology & labs, one row per hadm_id. labevents filtered
# to the locked blood itemids + in-stay gate (admittime <= charttime <= dischtime)
# + valuenum > 0. Per analyte: last/first(for delta)/max/min/n via
# conditional aggregation; measured = n>0. Missing stays NULL (a lab not drawn is
# not a 0) — imputation deferred to features_clean (train-only). In Dataform the
# spine reads FROM ${ref("cohort_split")} and the results FROM ${ref("labevents")}.
LAB_ITEMIDS = {
    "rbc":       51279,  # Red Blood Cells, Blood (m/uL)    ~89.6% coverage
    "rdw":       51277,  # RDW, Blood (%)                   ~89.6%
    "glucose":   50931,  # Glucose, Blood (mg/dL)           ~85.8%
    "monocytes": 52074,  # Absolute Monocyte Count (K/uL)   ~16.4% (differential CBC)
}

_itemid_list  = ", ".join(str(i) for i in LAB_ITEMIDS.values())
_case_analyte = "\n".join(f"      WHEN {iid} THEN '{a}'" for a, iid in LAB_ITEMIDS.items())

# Inner per-analyte aggregates (last/first/max/min/n), conditional on analyte.
_agg_block = ",\n".join(
    f"""    ARRAY_AGG(IF(analyte='{a}', valuenum, NULL) IGNORE NULLS ORDER BY charttime DESC LIMIT 1)[SAFE_OFFSET(0)] AS {a}_last,
    ARRAY_AGG(IF(analyte='{a}', valuenum, NULL) IGNORE NULLS ORDER BY charttime ASC  LIMIT 1)[SAFE_OFFSET(0)] AS {a}_first,
    MAX(IF(analyte='{a}', valuenum, NULL)) AS {a}_max,
    MIN(IF(analyte='{a}', valuenum, NULL)) AS {a}_min,
    COUNTIF(analyte='{a}')                 AS {a}_n"""
    for a in LAB_ITEMIDS
)

# Outer feature columns: last/max/min, delta = last - first, measured = n>0.
_out_block = ",\n".join(
    f"""  agg.{a}_last                  AS {a}_last,
  agg.{a}_max                   AS {a}_max,
  agg.{a}_min                   AS {a}_min,
  agg.{a}_last - agg.{a}_first  AS {a}_delta,
  COALESCE(agg.{a}_n, 0) > 0    AS {a}_measured"""
    for a in LAB_ITEMIDS
)

feat_labs_sql = BASE_CTE + f""",
spine AS (
  SELECT hadm_id, admittime, dischtime
  FROM flagged
  WHERE anchor_age >= 18
    AND hospital_expire_flag = 0 AND deathtime IS NULL
    AND los_hours >= 24
    AND admission_type NOT IN ({_planned_list})
    AND NOT is_continuation
),
le AS (
  -- In-stay numeric results for the locked itemids only, tagged by analyte.
  SELECT
    s.hadm_id,
    CASE le.itemid
{_case_analyte}
    END AS analyte,
    le.charttime,
    le.valuenum
  FROM spine AS s
  JOIN `{MIMIC_HOSP}.labevents` AS le
    ON le.hadm_id = s.hadm_id
   AND le.charttime >= s.admittime
   AND le.charttime <= s.dischtime
   AND le.itemid IN ({_itemid_list})
   AND le.valuenum > 0          -- all four analytes are strictly positive;
                                -- a 0/negative is a junk reading -> not measured
),
agg AS (
  SELECT
    hadm_id,
{_agg_block}
  FROM le
  GROUP BY hadm_id
)
SELECT
  s.hadm_id,
{_out_block}
FROM spine AS s
LEFT JOIN agg USING (hadm_id)
"""

feat_labs = run_sql(feat_labs_sql)
print(f"feat_labs rows: {len(feat_labs):,}  cols: {feat_labs.shape[1]}")
feat_labs.head()


feat_labs rows: 352,699  cols: 21


,hadm_id,rbc_last,rbc_max,rbc_min,rbc_delta,rbc_measured,rdw_last,rdw_max,rdw_min,rdw_delta,...,glucose_last,glucose_max,glucose_min,glucose_delta,glucose_measured,monocytes_last,monocytes_max,monocytes_min,monocytes_delta,monocytes_measured
0,26033450,2.94,4.56,2.61,-1.42,True,17.7,17.7,13.4,4.3,...,93.0,285.0,91.0,-192.0,True,0.19,0.19,0.19,0.0,True
1,20843203,4.17,4.17,4.17,0.00,True,11.7,11.7,11.7,0.0,...,74.0,74.0,74.0,0.0,True,NaN,NaN,NaN,NaN,False
2,24641199,3.65,3.65,3.52,0.13,True,20.3,20.3,13.9,6.4,...,80.0,106.0,80.0,-26.0,True,NaN,NaN,NaN,NaN,False
3,28152599,4.39,4.56,4.03,0.07,True,11.6,11.7,11.6,0.0,...,95.0,133.0,95.0,-38.0,True,NaN,NaN,NaN,NaN,False
4,22948797,2.95,3.44,2.95,-0.49,True,18.4,18.8,18.4,-0.4,...,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,False


In [24]:
# Step 3 — Sanity check. Grain integrity + the NULL contract (this is the first
# group where NULL is legitimate) + plausibility. Unlike the count groups, the
# guard is NOT "zero nulls" — it's "value cols are NULL iff the analyte wasn't
# measured", so nothing leaks a fake reading and nothing drops a real one.
l = feat_labs

ANALYTES = list(LAB_ITEMIDS)               # rbc, rdw, glucose, monocytes
EXPECTED_COV = {"rbc": 89.6, "rdw": 89.6, "glucose": 85.8, "monocytes": 16.4}  # from locate

# (a) one row per hadm_id, matching the spine (the agg GROUP BY must stay 1:1)
assert len(l) == 352_699, f"row count {len(l)} != spine 352,699"
assert l.hadm_id.is_unique, "hadm_id is not unique — the labevents join fanned out"

for a in ANALYTES:
    meas = l[f"{a}_measured"]
    # measured is a clean boolean (no nulls of its own)
    assert meas.notnull().all(), f"{a}_measured has nulls"
    for suf in ["last", "max", "min", "delta"]:
        col = l[f"{a}_{suf}"]
        # (b) NULL contract: value present iff measured, absent iff not measured
        assert col[meas].notnull().all(),  f"{a}_{suf} is NULL on a measured row"
        assert col[~meas].isnull().all(),  f"{a}_{suf} is non-NULL on an unmeasured row"
    # (c) ordering must hold on measured rows: min <= last <= max
    mr = l[meas]
    assert (mr[f"{a}_min"] <= mr[f"{a}_last"]).all(), f"{a}: last < min"
    assert (mr[f"{a}_last"] <= mr[f"{a}_max"]).all(), f"{a}: last > max"
    # (d) values are physiologically positive
    assert (mr[f"{a}_min"] > 0).all(), f"{a}_min has a non-positive value"

print(f"rows            : {len(l):,}  (unique hadm_id: {l.hadm_id.is_unique})")
print("\nmeasured coverage (sanity vs locate):")
for a in ANALYTES:
    cov = l[f"{a}_measured"].mean() * 100
    print(f"  {a:<10} {cov:5.1f}%   (locate: ~{EXPECTED_COV[a]}%)")

print("\n_last distribution per analyte (measured rows only):")
print(l[[f"{a}_last" for a in ANALYTES]].describe().round(2).to_string())


rows            : 352,699  (unique hadm_id: True)

measured coverage (sanity vs locate):
  rbc         89.6%   (locate: ~89.6%)
  rdw         89.6%   (locate: ~89.6%)
  glucose     85.8%   (locate: ~85.8%)
  monocytes   16.4%   (locate: ~16.4%)

_last distribution per analyte (measured rows only):
        rbc_last   rdw_last  glucose_last  monocytes_last
count  316150.00  316079.00     302512.00        57731.00
mean        3.70      14.96        116.09            0.74
std         0.70       2.34         43.70            0.73
min         1.32      10.50          5.00            0.01
25%         3.18      13.30         91.00            0.42
50%         3.69      14.40        103.00            0.64
75%         4.18      16.00        126.00            0.93
max         8.28      40.00       2613.00           60.21


### 3f. Triage & Chief Complaint (ED arrival state)

Looking at `mimiciv_ed.triage` as a candidate source. `triage` keys on `stay_id`
(the ED visit), so the link to the cohort is
`triage.stay_id → edstays.stay_id → edstays.hadm_id`.

**Step 1 — Locate.** First, the only question that matters right now: how many
cohort admissions actually have a `chiefcomplaint` value? Triage rows only exist
for ED-sourced admissions, so this is the coverage ceiling for any triage feature.


In [29]:
# Step 1 — Locate. One question: how many cohort patients have a chiefcomplaint
# value in mimiciv_ed.triage? Link is triage.stay_id -> edstays.stay_id ->
# edstays.hadm_id (the cohort key).
chiefcomplaint_coverage_sql = BASE_CTE + f""",
spine AS (
  SELECT hadm_id
  FROM flagged
  WHERE anchor_age >= 18
    AND hospital_expire_flag = 0 AND deathtime IS NULL
    AND los_hours >= 24
    AND admission_type NOT IN ({_planned_list})
    AND NOT is_continuation
)
SELECT
  (SELECT COUNT(*) FROM spine) AS cohort_admissions,
  COUNT(DISTINCT e.hadm_id)    AS admissions_with_chiefcomplaint
FROM spine AS s
JOIN `{MIMIC_ED}.edstays` AS e USING (hadm_id)
JOIN `{MIMIC_ED}.triage`  AS t USING (stay_id)
WHERE t.chiefcomplaint IS NOT NULL AND TRIM(t.chiefcomplaint) <> ''
"""
cc_coverage = run_sql(chiefcomplaint_coverage_sql)
cohort = int(cc_coverage["cohort_admissions"].iloc[0])
with_cc = int(cc_coverage["admissions_with_chiefcomplaint"].iloc[0])
print(f"cohort admissions                : {cohort:,}")
print(f"with a chiefcomplaint value      : {with_cc:,}  ({with_cc / cohort * 100:.1f}%)")


cohort admissions                : 352,699
with a chiefcomplaint value      : 142,301  (40.3%)


In [30]:
# Step 1 (cont.) — Cardinality + head concentration of chiefcomplaint, over the
# cohort's triage rows. This is the number that decides one-hot vs embedding:
#  - distinct normalized values  -> how wide a one-hot would be
#  - % covered by the top 20/50  -> whether a top-N one-hot captures most of it
# Normalization is just lowercase + trim (no token splitting yet).
chiefcomplaint_card_sql = BASE_CTE + f""",
spine AS (
  SELECT hadm_id
  FROM flagged
  WHERE anchor_age >= 18
    AND hospital_expire_flag = 0 AND deathtime IS NULL
    AND los_hours >= 24
    AND admission_type NOT IN ({_planned_list})
    AND NOT is_continuation
),
cc AS (
  SELECT LOWER(TRIM(t.chiefcomplaint)) AS chief_complaint
  FROM spine AS s
  JOIN `{MIMIC_ED}.edstays` AS e USING (hadm_id)
  JOIN `{MIMIC_ED}.triage`  AS t USING (stay_id)
  WHERE t.chiefcomplaint IS NOT NULL AND TRIM(t.chiefcomplaint) <> ''
),
ranked AS (
  SELECT chief_complaint, COUNT(*) AS n,
         ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC) AS rnk
  FROM cc
  GROUP BY chief_complaint
)
SELECT
  (SELECT COUNT(*) FROM cc)                                   AS total_rows,
  (SELECT COUNT(*) FROM ranked)                               AS distinct_values,
  (SELECT SUM(n) FROM ranked WHERE rnk <= 20)                AS top20_rows,
  (SELECT SUM(n) FROM ranked WHERE rnk <= 50)                AS top50_rows
"""
cc_card = run_sql(chiefcomplaint_card_sql)
r = cc_card.iloc[0]
total = int(r["total_rows"])
print(f"triage rows with a chiefcomplaint : {total:,}")
print(f"distinct normalized values        : {int(r['distinct_values']):,}")
print(f"top 20 values cover               : {int(r['top20_rows'])/total*100:5.1f}%")
print(f"top 50 values cover               : {int(r['top50_rows'])/total*100:5.1f}%")


triage rows with a chiefcomplaint : 142,638
distinct normalized values        : 27,015
top 20 values cover               :  26.4%
top 50 values cover               :  35.9%
